---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-12: Accesing OS LLMs using HF Transformers Pipeline</h1>

# Learning agenda of this notebook
1. Recap: Ways to Access Open Source LLMs
2. Hugging Face Developer Ecosystem and Libraries
    - `transformers`
    - `peft`
    - `trl`
    - `unsloth`
3. The `transformers.pipeline()` Method
    - Key Parameters to `transformers.pipeline()` Method
    - Key Parameters to `pipe-object()` Method
4. Hands-On Examples of Performing Basic NLP Tasks using `transformer.pipeline()` Method
    - Example 1: Sentiment Analysis, Text Classification, and Zero-Shot- Classification
    - Example 2: Fill Mask Pipeline
    - Example 3: Text Generation
    - Example 4: Named Entity Recognition
    - Example 5: Summarization
    - Example 6: Translation
    - Example 7: Question Answering
5. Using Chat Templates with `transformer.pipeline()` Method
    - Example 1
    - Example 2
    - Example 3
    - Example 4

# <span style='background :lightgreen' >1. Recap: Ways to Access Open Source LLMs</span>

### (i) Access Open-Source Models via Cloud-Based Providers (Driving a fully automatic car — everything managed for you): Groq, SambaNova, TogetherAI, Replicate, Fireworks-ai, Cerebras

### (ii) Run Open-Source Models locally using runtimes (Driving an automatic car — local but simple, no gears or engineering): llama.cpp, ollama, LM Studio, GPT4All

### (iii) Use Open-Source Models via Hugging Face `pipeline()` API (Driving a manual car — you see more of the mechanics, but still a car someone else built)


### (iv) Load and run models directly from Hugging Face Hub using `AutoModel/AutoTokenizer` (Opening the hood and adjusting or replacing engine components)


### (v) Fine-Tune LLMs using full fine-tuning or PEFT methods (LoRA / QLoRA / adapters) (Upgrading and re-calibrating the engine to suit your driving style)

### (vi) Build and train an AI Model from scratch using PyTorch / TensorFlow (Designing and building the entire car from raw parts — full control, full responsibility)


# <span style='background :lightgreen' >2. Hugging Face Developer Ecosystem and Libraries</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The Hugging Face ecosystem consists of multiple libraries that work together to make training, fine-tuning, and deploying AI models easier.</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The single most important library to understand is <b>Transformers</b>, every other library in this echosystem (peft, trl, unsloth) is built on top of it.</h3>

## a. **`tramsformers`**: Core Library
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Hugging Face Transformers is a library that provides tools for <b>using, loading, fine-tuning,</b> and <b>pre-training</b> open-source AI models with a unified API that works seamlessly with Meta's PyTorch and Google's TensorFlow/Keras and JAX.</h3>

### Components that are part of transformers package
- **Model Hub Integration:** The Hugging Face Model Hub is an online platform hosting millions of models, datasets, and spaces. It enables easy downloading, uploading, versioning, sharing, and discovering of models and datasets. Beyond loading models, the `huggingface_hub` library also provides a programmatic API (`HfApi`) to interact with the Hub directly (to list, search, delete, or manage your uploaded models):
```python
from huggingface_hub import login, HfApi
login(token=hf_token)                        
client = HfApi()                             
models_gen = client.list_models(author="arif-butt") 
models     = list(models_gen)                       
for model in models:
    print(model.id)             
```

<br>

- **Pipeline** The simplest way to use models. It's a high-level abstraction that wraps the entire ML workflow (tokenization, model inference, and decoding) into a single function call, allowing developers to perform tasks like classification, translation, summarization, and question answering with minimal code. It supports 30+ tasks out of the box
```python
from transformers import pipeline
pipe = pipeline("text-generation")
print(pipe("Which is the highest mountain peak in Pakistan?")
```

<br>

- **Models** Neural network architectures that perform the actual computations. The Transformers library provides standardized implementations of modern text, vision, audio, and multimodal models, and includes AutoModel classes that automatically load the correct architecture for a chosen task. First call downloads and caches locally (`~/.cache/huggingface/`). Subsequent calls load from cache instantly — no re-download
```python
from transformers import  AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
```

<br>

- **Tokenizers:** Handle conversion between raw text and numerical tokens that models can process. They split text into subword tokens, map them to integer IDs, and decode model outputs back to readable text. Every model has its own vocabulary and tokenization rules, the `AutoTokenizer` automatically loads the correct one.
```python
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
```



- **Config:** A Config object defines a model's architecture settings, such as number of layers, hidden sizes, attention heads, dropout, vocabulary size, context length etc. Every model on HuggingFace Hub has a `config.json` configuration file that ensures the correct architecture is instantiated when loading a model:
```python
from transformers import AutoConfig
config = AutoConfig.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
```

<br>

- **Trainer:** The Trainer API provides a complete, high-level training loop for fine-tuning transformer models. It manages batching, gradient accumulation, mixed precision (fp16/bf16), learning rate scheduling, evaluation, logging, and checkpointing. Works with any `transformers` model and standard PyTorch datasets
```python
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(...)
trainer = Trainer(
                model         = model,
                args          = training_args,
                train_dataset = train_dataset,
                eval_dataset  = eval_dataset,
                )
trainer.train()
```
>- **Limitation:**  `Trainer` is a general training loop as it does not natively support LoRA, response-only loss masking, or preference training. That is where `peft` and `trl` come in.

## b. **`peft`** (Parameter Efficient Fine-Tuning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>peft</b> is a HF library used to fine-tune transformer-based models by injecting small trainable adapter modules while keeping the massive base model weights completely <b>frozen</b>. It is designed exclusively for <b>Supervised Fine-Tuning (SFT)</b> and cannot perform pre-training, continued pre-training, or alignment on its own.</div></h3>

## c. **`trl`** (Transformer Reinforcement Learning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>trl</b> is a HF library that provides complete training pipelines for every <b>post-pretraining stage</b> of LLM development (SFT, DPO, GRPO, PPO, RLHF and reward modeling) built on top of transformers and peft. (pre-training is outside its scope)</div></h3>

## d. **`unsloth`** (2–5× Faster Fine-Tuning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>Unsloth</b> is an open-source library that makes LoRA and QLoRA fine-tuning <b>2–5× faster</b> and uses <b>60% less VRAM</b> than the standard transformers + peft + trl stack,  by rewriting critical GPU kernels in Triton, with zero loss in accuracy. It supports <b>SFT and alignment</b> but is not designed for pre-training or continued pre-training on raw text corpora.</div></h3>



## 📊 Comparison of all Four Libraries

<pre>
┌─────────────────────────────────────────────────────────────────┐
│                        <b>unsloth</b>                                  │
│   2-5× faster kernels, 60% less VRAM, GGUF export               │
│      (Wraps and accelerates everything below)                   │
├────────────────────────────┬────────────────────────────────────┤
│          <b>peft</b>              │                 <b>trl</b>                │
│  LoRA, QLoRA, Adapters     │  SFTTrainer, DPOTrainer            │
│  Freeze base model         │  GRPOTrainer, PPOTrainer           │
│  Train only adapters       │  RewardTrainer                     │
├────────────────────────────┴────────────────────────────────────┤
│                    <b>transformers</b>                                 │
│   Models, Tokenizers, Configs, Pipelines, Trainer               │
│   Feature Extractors, Processors, Hub Integration               │
├─────────────────────────────────────────────────────────────────┤
│              PyTorch / TensorFlow / Keras / JAX                 │
│    (DL frameworks on which above libraries are built)           │
└─────────────────────────────────────────────────────────────────┘
</pre>

# <span style='background :lightgreen' >3. The Transformers Pipeline</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The HuggingFace transformer pipeline is the easiest way to use a pre-trained model for a given task.</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">It is an end-to-end workflow for NLP tasks that handles text preprocessing, tokenization, loading pre-trained transformer models, and generating predictions.</h3>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The `transformers.pipeline()` method wraps the AI model inference workflow into a single call, allowing developers to perform tasks like classification, translation, summarization, or question answering with minimal code.</h3>

<div style="text-align:center;">
    <img src="../images/pipeline-abstract.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>

##  a. The `transformers.pipeline()` Method

- The `transformers.pipeline()` method is the most hight level API of the transformers library that automatically handles tokenization, model loading, preprocessing, inference, and postprocessing.
- It lets you go from raw input (text / image / audio, depending on pipeline type) to human‑readable predictions in a single call.
- Provides a unified, high-level interface for running text, vision, audio, and multimodal models without explicitly dealing with tokenizers or model architectures.
- Supports a broad range of tasks like text generation, classification, question answering, image analysis, audio processing, and more through one consistent abstraction.
- Enables instant loading of thousands of models from the Hugging Face Hub, supports GPU and Apple-Silicon acceleration, and can run entirely on local hardware for privacy.
- Allows access to gated models (e.g., Llama-3.1-8B-Instruct,  meta-llama/Llama-4-Maverick-17B-128E-Instruct) simply by providing an access token after accepting their usage terms.
- It is as simple as the following LOCs:
```python
from transformers import pipeline
pipe = pipeline("sentiment-analysis") # Use default model, will return a pipeline object of the task-specific pipeline class
result = pipe("I love using Hugging Face!")  # Format of `result` depends on the task you selected and sometimes on the model.
```
- **Workflow:**
```markdown
pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")
│
├── STEP 1: Downloads model weights          ┐
├── STEP 2: Downloads model config           ├── This is what AutoModelForCausalLM.from_pretrained() also do (ONLY this part)
├── STEP 3: Downloads tokenizer files        │   
├── STEP 4: Loads everything into memory     ┘
│
├── STEP 5: Wraps model + tokenizer into a pipeline object
├── STEP 6: Pipeline object handles preprocessing, i.e., tokenization of your input text automatically
├── STEP 7: Pipeline object runs model.generate() for you to get logits or probabilities
└── STEP 8: Pipeline object handles postprocessing, i.e., decodes output tokens back to text automatically
```

- **First run:** Hugging Face will download following files from the Hub, which are cached under `~/.cache/huggingface/hub/` (Linux/macOS) or `%USERPROFILE%/.cache/huggingface/hub/` (Windows). :
    - Model weights (`pytorch_model.bin` or `model.safetensors`)
    - Model configuration (`config.json`)
    - Tokenizer files (`tokenizer.model`, `tokenizer.json`, `tokenizer_config.json`, `vocab.json`, `special_tokens_map.json`)
- **Subsequent runs:** No internet download → files are loaded from local disk cache. Only the move from disk → CPU RAM → GPU VRAM happens, which can still take a second or two. Inference is done locally.
- **After reboot:** No re-download, unless you manually clear your cache or set `force_download=True`.
- **What does not happen:**
    - ❌ No API calls are made to a hosted Hugging Face inference endpoint
    - ❌ Your input string is not sent over the network
    - ❌ No Hugging Face API key is required

## b. Key Parameters to `transformers.pipeline()` Method
```python
from transformers import pipeline
pipe = pipeline(
    task="sentiment-analysis",           # REQUIRED: the task type (e.g., "sentiment-analysis", "text-generation", "summarization", "question-answering", etc.)
    model=None,                          # OPTIONAL: specific model name or local path; if None, pipeline selects a default model for the task
    tokenizer=None,                      # OPTIONAL: specific tokenizer name or local path; if None, inferred from the model
    framework=None,                      # OPTIONAL: deep learning backend ('pt' for PyTorch, 'tf' for TensorFlow); default is inferred from model
    device=-1,                           # OPTIONAL: device to run the model on; -1 = CPU, >=0 = GPU device index
    batch_size=1,                        # OPTIONAL: number of inputs to process in a batch; default 1
    use_fast=True,                       # OPTIONAL: whether to use fast tokenizer if available; default True
    return_all_scores=False,             # OPTIONAL: for classification tasks, return scores for all labels; default False
    function_to_apply="auto",            # OPTIONAL: specify 'none', 'text', or 'sigmoid' post-processing; default "auto"
    max_length=None,                     # OPTIONAL: override model's default max length
    return_full_text=False,              # OPTIONAL: By setting this to False, the prompt will not be returned but merely the output of the model.
    top_k=50,                            # OPTIONAL: for generation tasks, number of highest-probability tokens to consider; default 50
    top_p=1.0,                           # OPTIONAL: for nucleus sampling in generation tasks; default 1.0
    **kwargs                             # OPTIONAL: any other task-specific parameters (e.g., num_beams, temperature, min_length, etc.)
)
```

### (i) The `task` Argument:

| Task | Modality | Pipeline Identifier | Description |
|------|----------|-------------------|-------------|
| Sentiment Analysis | NLP | `pipeline(task="sentiment-analysis")` | Determine the emotional tone or opinion expressed in text (positive, negative, neutral). Often used for social media monitoring, customer feedback analysis, and brand sentiment tracking. **Model size:** ~250 MiB. **Example input:** `"This product is terrible"` |
| Text Classification | NLP | `pipeline(task="text-classification")` | Classify text into predefined categories such as topic classification, spam detection, intent recognition, or content moderation. Returns confidence scores for each possible class label. **Model size:** ~250-500 MiB. **Example input:** `"Breaking news: Scientists discover new planet"` |
| Zero-Shot Classification | NLP | `pipeline(task="zero-shot-classification")` | Classify text into categories without prior training on those specific labels. You provide candidate labels, and the model determines which best fits the input text using natural language understanding. **Model size:** ~400-1.3 GiB. **Example input:** `{"sequences": "This is a sports article", "candidate_labels": ["politics", "sports", "technology"]}` |
| Fill Mask Pipeline | NLP | `pipeline(task="fill-mask")` | Predict missing words in text where tokens are masked (replaced with [MASK]). Useful for text completion, spell checking, and understanding contextual word relationships in sentences. **Model size:** ~400-500 MiB. **Example input:** `"The capital of France is [MASK]"` |
| Text Generation | NLP | `pipeline(task="text-generation")` | Generate coherent text continuations given an input prompt. Can be used for creative writing, code completion, chatbots, or story generation. Supports temperature and length controls for output variation. **Model size:** ~500 MiB-13 GiB. **Example input:** `"Once upon a time in a distant galaxy"` |
| Named Entity Recognition | NLP | `pipeline(task="ner")` | Identify and classify named entities in text such as people, organizations, locations, dates, and custom entity types. Essential for information extraction, document processing, and knowledge graph construction. **Model size:** ~400-500 MiB. **Example input:** `"Apple Inc. was founded by Steve Jobs in Cupertino"` |
| Summarization | NLP | `pipeline(task="summarization")` | Create concise summaries of longer documents or articles while preserving key information. Supports both extractive (selecting sentences) and abstractive (generating new text) summarization approaches. **Model size:** ~900 MiB-3 GiB. **Example input:** `"Long article text about climate change research findings..."` |
| Translation | NLP | `pipeline(task="translation")` | Convert text from one language to another while preserving meaning and context. Supports many language pairs and can handle various text types from formal documents to casual conversations. **Model size:** ~600 MiB-2 GiB. **Example input:** `"Hello, how are you today?"` |
| Question Answering | NLP | `pipeline(task="question-answering")` | Extract answers from a given context/passage based on natural language questions. The model identifies the relevant span of text that best answers the question within the provided document. **Model size:** ~400-500 MiB. **Example input:** `{"question": "What is AI?", "context": "Artificial Intelligence is..."}` |
| Audio classification | Audio | `pipeline(task="audio-classification")` | Classify audio clips into categories such as music genres, environmental sounds, speech emotions, or spoken language identification. Processes raw audio waveforms or spectrograms. **Model size:** ~300-600 MiB. **Example input:** Audio file path or numpy array |
| Automatic speech recognition | Audio | `pipeline(task="automatic-speech-recognition")` | Convert spoken language into written text with high accuracy. Supports multiple languages, real-time transcription, and can handle various audio formats and quality levels. **Model size:** ~240 MiB-3 GiB. **Example input:** Audio file `"speech.wav"` or audio array |
| Text-to-speech | Audio | `pipeline(task="text-to-speech")` | Convert written text into natural-sounding speech audio. Supports multiple voices, languages, and speaking styles. Requires speaker embeddings for voice characteristics and uses vocoder models for high-quality audio synthesis. **Model size:** ~400 MiB-1 GiB. **Example input:** `"Hello, welcome to our presentation"` |
| Image classification | Computer Vision | `pipeline(task="image-classification")` | Categorize images into predefined classes (e.g., animals, objects, scenes). Returns top predictions with confidence scores. Useful for content moderation, photo organization, or automated tagging. **Model size:** ~80-500 MiB. **Example input:** Image file `"cat.jpg"` or PIL Image object |
| Image segmentation | Computer Vision | `pipeline(task="image-segmentation")` | Identify and label every pixel in an image. Supports semantic segmentation (classify pixels), instance segmentation (separate individual objects), and panoptic segmentation (combine both approaches). **Model size:** ~200 MiB-1 GiB. **Example input:** Image file or PIL Image object |
| Object detection | Computer Vision | `pipeline(task="object-detection")` | Locate and classify multiple objects within an image, providing bounding box coordinates and class labels. Essential for autonomous vehicles, surveillance, and inventory management systems. **Model size:** ~160-800 MiB. **Example input:** Image file `"street_scene.jpg"` or PIL Image |
| Visual question answering | Multimodal | `pipeline(task="vqa")` | Answer natural language questions about the content of images by combining computer vision and NLP. Can identify objects, count items, describe relationships, and understand spatial concepts. **Model size:** ~1-2 GiB. **Example input:** `{"image": "photo.jpg", "question": "How many cats are in this image?"}` |
| Document question answering | Multimodal | `pipeline(task="document-question-answering")` | Extract answers from documents (PDFs, images with text) by combining OCR, layout understanding, and reading comprehension. Handles forms, invoices, contracts, and structured documents. **Model size:** ~1-3 GiB. **Example input:** `{"image": "invoice.pdf", "question": "What is the total amount?"}` |
| Image captioning | Multimodal | `pipeline(task="image-to-text")` | Generate natural language descriptions of image content, including objects, actions, scenes, and relationships. Useful for accessibility, content generation, and image indexing applications. **Model size:** ~1-2 GiB. **Example input:** Image file `"landscape.jpg"` or PIL Image object |

## (ii) The  `model` Argument
```python
from transformers import pipeline
pipe = pipeline("sentiment-analysis") # Use default model
pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest") # Use specific model from Hub by specifing model ID
pipe = pipeline("text-classification", model="./my-local-model") # Use locally saved model
pipe = pipeline("text-classification", model="your-username/your-model-name") # Use model you uploaded to Hugging Face Hub
```

### Default Models by Task

| Task (Pipeline Name)         | Default Model                                     | Size/Notes |
| ---------------------------- | ------------------------------------------------- | ---------- |
| `sentiment-analysis`         | `distilbert-base-uncased-finetuned-sst-2-english` | ~250 MB |
| `text-generation`            | `gpt2`                                            | ~548 MB |
| `question-answering`         | `distilbert-base-cased-distilled-squad`           | ~250 MB |
| `ner`                        | `dslim/bert-base-NER`                             | ~440 MB |
| `summarization`              | `sshleifer/distilbart-cnn-12-6`                   | ~220 MB |
| `translation_en_to_de`       | `Helsinki-NLP/opus-mt-en-de`                      | ~120 MB |
| `zero-shot-classification`   | `facebook/bart-large-mnli`                        | ~1.6 GB |
| `text2text-generation`       | `google/flan-t5-base`                             | ~990 MB |
| `conversational`             | `microsoft/DialoGPT-medium`                       | ~863 MB |

- While executing a pipeline, you may get an error saying access to the specific model is restricted and you are not in the authorized list. If this is the case generate a request to access to the Model as mentioned below:
    - Visit the model page e.g., https://huggingface.co/black-forest-labs/FLUX.1-schnell
    - Click the "Request access" button
    - Fill out the access request form
    - Wait for approval (can take hours to days)
    - Finally you will get this message on the model page: **Gated model: You have been granted access to this model**
- Visit https://huggingface.co/settings/gated-repos, in order to view the gated repositories that you have requested access to

## (iii) The `tockenizer` Argument:
- The tokenizer argument lets you specify or override the tokenizer used with the model. Most of the time, Hugging Face automatically loads the right tokenizer based on the model, but you can explicitly choose one — for speed, compatibility, or custom behavior.

| Tokenizer Name  | Typical Use / Model Family | Notes                          |
| ----------------------------------- | -------------------------- | ------------------------------ |
| `"bert-base-uncased"`               | BERT, DistilBERT           | Lowercased input               |
| `"distilbert-base-uncased"`         | DistilBERT                 | Lightweight BERT               |
| `"gpt2"`                            | GPT-2, GPT-Neo             | No padding token by default    |
| `"t5-base"`                         | T5                         | Text-to-text tasks             |
| `"openai/whisper-small"`            | Whisper                    | Speech recognition             |
| `"microsoft/phi-2"`                 | Phi models                 | Good for compact LLM inference |
| `"meta-llama/Meta-Llama-3-8B"`      | LLaMA 2 / 3                | Must use with `AutoTokenizer`  |
| `"google/gemma-2b"`                 | Gemma models               | Decoder-only LLMs              |


## c. Key Parameters to `pipe-object()` Method
- When you call `transformers.pipeline(...)` method, it returns an instance of a task-specific pipeline class, to which you can pass input(s) to get results.
- The returned pipeline object works out-of-the box for common tasks like sentiment-analysis, text-classification, text-generation, summarization, translation etc.
- The returned pipeline object has a standardized callable interface that wraps all the steps for:
    - Preprocessing: Converts raw text → token IDs using the tokenizer.
    - Model Inference: Passes token IDs through the model to get logits or probabilities.
    - Postprocessing: Converts model outputs → human-readable labels and scores.
```python
output = pipe(
    "Your input text here",        # REQUIRED: the input to process (text, list of texts, question/context, etc. depending on task). Always pass the input as positional — never as keyword
    max_length=None,               # OPTIONAL: maximum sequence length for generation or truncation; default None
    min_length=None,               # OPTIONAL: minimum length for generation tasks
    do_sample=False,               # OPTIONAL: whether to use sampling for text generation; default False (greedy)
    temperature=1.0,               # OPTIONAL: controls randomness of generation; default 1.0
    top_k=50,                      # OPTIONAL: number of highest-probability tokens to consider in generation; default 50
    top_p=1.0,                     # OPTIONAL: cumulative probability threshold for nucleus sampling; default 1.0
    num_return_sequences=1,        # OPTIONAL: number of sequences to generate per input; default 1
    return_full_text=True,         # OPTIONAL: return the full prompt along with generated text; default True
    truncation=True,               # OPTIONAL: truncate sequences longer than model max length; default True
    padding=True,                  # OPTIONAL: pad sequences to same length if required; default True
    return_tensors=None,           # OPTIONAL: return PyTorch or TensorFlow tensors instead of decoded text; default None
    function_to_apply="auto",      # OPTIONAL: for classification tasks, specify post-processing function ('text', 'sigmoid', 'none'); default 'auto'
    return_all_scores=False,       # OPTIONAL: for classification tasks, return scores for all labels; default False
    **kwargs                       # OPTIONAL: any other task-specific call-time parameters (e.g., num_beams, early_stopping, eos_token_id, pad_token_id)
)
```

# <span style='background :lightgreen' >4. Hands-On Examples of Performing Basic NLP Tasks using `transformer.pipeline()` Method</span>

## Example 1: Sentiment Analysis, Text Classification, and Zero-Shot- Classification
<div style="text-align:center">
    <img src="../images/pl1.png" width="1000">
</div>

### The `sentiment-analysis` pipeline
- The `sentiment-analysis` pipeline lassifies input text into sentiment categories (e.g., positive, negative, neutral).
- You can pass multiple texts to the same pipeline which will be processed and passed through the model together as a batch. The output will be a list of individual results in the same order as the input texts.
- After inference, the returned object is a list containing one dictionary per input. Each dictornary having:
    - label: the predicted class.
    - score: the probability/confidence of that label.

In [1]:
from transformers import pipeline
sa_pipeline = pipeline(task="sentiment-analysis")
sa_pipeline

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [9]:
# Now pass the input text to the sentiment analysis pipeline for processing.
sa_pipeline("I'm super excited to work on these awsome notebooks!")

[{'label': 'POSITIVE', 'score': 0.9800775051116943}]

In [10]:
# Now pass the input text to the sentiment analysis pipeline for processing.
sa_pipeline(["I'm super excited to work on these awsome notebooks!", "I hate this so much", "I am not going to university"])

[{'label': 'POSITIVE', 'score': 0.9800775051116943},
 {'label': 'NEGATIVE', 'score': 0.9995144605636597},
 {'label': 'NEGATIVE', 'score': 0.9996070265769958}]

### The `zero-shot-classification` pipeline
- The `zero-shot-classification` pipeline is a more general text classification pipeline.
- You can provide a set of candidate labels at runtime, and the model predicts which label best fits each input.
- After inference, the returned object is a dictionary having:
    - sequence: original input.
    - labels: candidate labels ranked by score.
    - scores: confidence for each label.

In [5]:
from transformers import pipeline
zsc_pipeline = pipeline("zero-shot-classification")
zsc_pipeline

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [11]:
# Now pass the input text to the sentiment analysis pipeline for processing.
zsc_pipeline("This lecture is about transformers library.", candidate_labels = ["politics", "sports", "education", "business"])

{'sequence': 'This lecture is about transformers library.',
 'labels': ['education', 'business', 'sports', 'politics'],
 'scores': [0.9395636320114136,
  0.03770514577627182,
  0.011649406515061855,
  0.01108181569725275]}

### The `text-classification` pipeline
- The `text-classification` pipeline is a general-purpose text classifier for any task where a model has been fine-tuned on specific labels (e.g., topics, spam detection, emotion).
- Works like sentiment-analysis but for custom or task-specific categories.
- After inference, the returned object is a dictionary having:
    - label
    - score

In [6]:
from transformers import pipeline
#  A multi-label classification model with 28 'probability' float outputs for any given input text. Typically a threshold of 0.5 is applied to the probabilities for the prediction for each label.
text_classifier_pipeline = pipeline("text-classification", model = "SamLowe/roberta-base-go_emotions")
text_classifier_pipeline

Device set to use mps:0


In [7]:
# Now pass the input text to the text classification pipeline for processing.
text_classifier_pipeline("I'm super excited to work on these awsome notebooks!")

[{'label': 'excitement', 'score': 0.8267171382904053}]

In [8]:
text_classifier_pipeline("I love the youtube videos of Arif")

[{'label': 'love', 'score': 0.9469678997993469}]

In [9]:
text_classifier_pipeline("Alas! no body understands what i am saying")

[{'label': 'disappointment', 'score': 0.5080626606941223}]

In [10]:
text_classifier_pipeline("It is heartbreaking, that we have lost the world cup of GenAI")

[{'label': 'sadness', 'score': 0.8933157324790955}]

## Example 2: Fill Mask Pipeline
- The fill mask pipeline is the pre-training objective of BERT, which is to guess the value of masked word.
- This is a sort of text completion.

In [1]:
from transformers import pipeline
fillmask_pipeline = pipeline(task="fill-mask")
fillmask_pipeline

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [2]:
# let us ask for the two most likely values for the missing word
fillmask_pipeline("Listen to your <mask>.", top_k=5)

[{'score': 0.09220784902572632,
  'token': 4312,
  'token_str': ' thoughts',
  'sequence': 'Listen to your thoughts.'},
 {'score': 0.09034585952758789,
  'token': 6456,
  'token_str': ' feedback',
  'sequence': 'Listen to your feedback.'},
 {'score': 0.06070244684815407,
  'token': 2236,
  'token_str': ' voice',
  'sequence': 'Listen to your voice.'},
 {'score': 0.030662477016448975,
  'token': 8591,
  'token_str': ' podcast',
  'sequence': 'Listen to your podcast.'},
 {'score': 0.02569151483476162,
  'token': 930,
  'token_str': ' music',
  'sequence': 'Listen to your music.'}]

In [3]:
# let us ask for the two most likely values for the missing word
fillmask_pipeline("I am going to Department of Data Science to <mask>.", top_k=5)

[{'score': 0.08922125399112701,
  'token': 1994,
  'token_str': ' speak',
  'sequence': 'I am going to Department of Data Science to speak.'},
 {'score': 0.06401623040437698,
  'token': 3922,
  'token_str': ' explain',
  'sequence': 'I am going to Department of Data Science to explain.'},
 {'score': 0.06287522614002228,
  'token': 1067,
  'token_str': ' talk',
  'sequence': 'I am going to Department of Data Science to talk.'},
 {'score': 0.06048249453306198,
  'token': 4830,
  'token_str': ' investigate',
  'sequence': 'I am going to Department of Data Science to investigate.'},
 {'score': 0.059621620923280716,
  'token': 16914,
  'token_str': ' enroll',
  'sequence': 'I am going to Department of Data Science to enroll.'}]

## Example 3: Text Generation
- The text generation pipeline, will auto-complete a given prompt
- The output is generated with a bit of randomness, so it changes each time you call the generator object on a given prompt.
- After inference, the returned object is a list of dictionaries for each input having a key named `generated_text` containing the generated string. For multiple generations per input, list contains multiple dictionaries per input (depends on num_return_sequences parameter).

In [14]:
from transformers import pipeline
text_generation_pipeline = pipeline("text-generation", model="distilgpt2",  device=-1)
text_generation_pipeline

Device set to use cpu


In [15]:
text_generation_pipeline("In this course, I will teach you how to",
                        max_length=30,
                        num_return_sequences=2
                        )

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In this course, I will teach you how to learn the tools to make a difference.\n\n\n\nIntroduction\nThe Basics\nIntroduction\nIn the above guide, you will learn a simple method to build your own virtual machine. You will first be able to use any of the following tools:\nSetup\nIn your setup, you will start from the start of your virtual machine, using any of the following tools:\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBox\nVirtualBo

## Example 4: Named Entity Recognition
- The `ner` pipeline perform token classification, i.e., identifies entities such as persons, organizations or locations in a sentence.

<img align="center" width="1000" src="../images/pl2.png"  > 

In [6]:
from transformers import pipeline
#ner_tagger = pipeline("ner", aggregation_strategy="simple", model="dslim/bert-base-NER")
ner_tagger = pipeline("ner", grouped_entities=True)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0
/Users/a

In [7]:
ner_tagger("Arif is a teacher at Punjab University, Lahore.")

[{'entity_group': 'PER',
  'score': np.float32(0.9948047),
  'word': 'Arif',
  'start': 0,
  'end': 4},
 {'entity_group': 'ORG',
  'score': np.float32(0.9882705),
  'word': 'Punjab University',
  'start': 21,
  'end': 38},
 {'entity_group': 'LOC',
  'score': np.float32(0.9970663),
  'word': 'Lahore',
  'start': 40,
  'end': 46}]

In [18]:
text = '''
Zelaid Mujahid is a sophomore majoring in Data Science at University of the Punjab. \
He is Pakistani national and has a 3.5 GPA. Mujahid is an active member of the department's AI Club.\
He hopes to pursue a career in AI after graduating.
'''
ner_tagger(text)

[{'entity_group': 'PER',
  'score': np.float32(0.99490315),
  'word': 'Zelaid Mujahid',
  'start': 1,
  'end': 15},
 {'entity_group': 'ORG',
  'score': np.float32(0.99170065),
  'word': 'University of the Punjab',
  'start': 59,
  'end': 83},
 {'entity_group': 'MISC',
  'score': np.float32(0.99775827),
  'word': 'Pakistani',
  'start': 91,
  'end': 100},
 {'entity_group': 'PER',
  'score': np.float32(0.9904875),
  'word': 'Mujahid',
  'start': 129,
  'end': 136},
 {'entity_group': 'ORG',
  'score': np.float32(0.8136327),
  'word': 'AI Club',
  'start': 177,
  'end': 184},
 {'entity_group': 'ORG',
  'score': np.float32(0.59545857),
  'word': 'AI',
  'start': 216,
  'end': 218}]

## Example 5: Summarization
- The `summarization` pipeline, help us in getting short summaries of very long articles.

In [19]:
from transformers import pipeline
summarizer = pipeline("summarization", device=-1)

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [20]:
text = """
Hugging Face is an open-source company and community widely recognized as the central hub for machine learning, especially natural language processing (NLP). It started with the development of Transformers, a popular Python library that provides easy access to state-of-the-art deep learning models such as BERT, GPT, and RoBERTa. Over time, Hugging Face has expanded into a full ecosystem for building, sharing, and deploying AI models. At the core of Hugging Face is the Hugging Face Hub, a collaborative platform where researchers, developers, and organizations can upload and download pre-trained models, datasets, and even demos. This drastically reduces the barrier to entry for machine learning, since users don’t need to train large models from scratch. Instead, they can leverage pre-trained models and fine-tune them for specific tasks like text classification, translation, summarization, or image recognition.
The company also supports tools beyond NLP, covering areas such as computer vision, speech recognition, and reinforcement learning. Features like the Inference API allow serverless model execution in the cloud, while AutoTrain automates model training.
Hugging Face has become a leader in the AI community by promoting openness, collaboration, and accessibility—making cutting-edge AI technology available to everyone from researchers to industry practitioners.
"""
result = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(result)
print(result[0]['summary_text'])

[{'summary_text': ' Hugging Face is an open-source company and community widely recognized as the central hub for machine learning . It started with the development of Transformers, a Python library that provides easy access to state-of-the-art deep learning models'}]
 Hugging Face is an open-source company and community widely recognized as the central hub for machine learning . It started with the development of Transformers, a Python library that provides easy access to state-of-the-art deep learning models


## Example 6: Translation

In [21]:
# All translation models are here: https://huggingface.co/models?pipeline_tag=translation&sort=trending
from transformers import pipeline
en_ur_translator = pipeline("translation_en_to_ur", model="Helsinki-NLP/opus-mt-en-ur", device=-1)
#pipe = transformers.pipeline("translation_en_to_fr", device='cuda')       # english to french
#pipe = transformers.pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device='cuda')

/Users/arif/Documents/genai-course/.venv/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


In [22]:
result = en_ur_translator("Learning is fun with Arif.")
print(result[0]['translation_text'])

سیکھنے کا مذاق ہے.


In [23]:
result = en_ur_translator("The budget this year will have a very bad impact on the low salried people.")
print(result[0]['translation_text'])

اس سال حساب کم ہونے والے لوگوں پر بہت ہی برا اثر پڑے گا.


In [24]:
result = en_ur_translator("I shot an elephant wearing pajamas.")
print(result[0]['translation_text'])

میں نے ایک ہاتھی کو گولی مار دی.


## Example 7: Question Answering
<img align="center" width="1000" src="../images/pl3.png"  > 

- The `question-answering` pipeline extracts answers to a question from a given text. from 🤗 Transformers expects two things:
    - context → a passage of text
    - question → a query about that passage
- The model then extracts the most relevant answer from the context and returns a single dictionary (not a list) with keys:
    - score: confidence.
    - startt and end: token positions in the context.
    - answer: predicted answer span.

In [25]:
from transformers import pipeline
qa_pipeline = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [26]:
result = qa_pipeline(question="What is AI?", 
           context="I am a teacher and I am learning Artificial Intelligence these days, however, I do teach Operating system as well as ...",
           max_answer_len=30,    # Maximum answer length
           max_seq_len=512,      # Maximum total sequence length
           doc_stride=128)       # Overlap when splitting long contexts
print(result)
print(result['answer'])

{'score': 0.745926558971405, 'start': 33, 'end': 56, 'answer': 'Artificial Intelligence'}
Artificial Intelligence


# <span style='background :lightgreen' >5. Using Chat Templates with `pipe()` Method</span>

| Aspect | Simple `pipe("text")` | Chat Template `apply_chat_template()` + `pipe(prompt)` |
|---|---|---|
| **Syntax** | `pipe("What is ML?")` | `prompt = pipe.tokenizer.apply_chat_template(messages, ...)` then `pipe(prompt)` |
| **Input format** | Raw plain text string | List of role-based message dictionaries |
| **What model sees** | `"What is ML?"` — raw text | `<\|system\|>...<\|user\|>...<\|assistant\|>` — structured special tokens |
| **System prompt support** | ❌ No | ✅ Yes — set model persona and behavior |
| **Multi-turn conversations** | ❌ No | ✅ Yes — include full conversation history |
| **Model name clues** | No special suffix | Contains **"Instruct"**, **"Chat"**, **"-it"** |
| **Example models** | `GPT-2`, `TinyLlama-1.1B` | `Llama-3.2-1B-Instruct`, `TinyLlama-1.1B-Chat`, `Mistral-7B-Instruct` |
| **Without correct approach** | Works fine — model completes text naturally | Model may continue text randomly — does not know where to start responding |
| **Tasks** | Classification, Sentiment, NER, Translation, Summarization, Simple text completion | Conversational AI, Custom personas, Production chatbots, Instruction following |

> 💡 **Rule of thumb:** If the model name contains **"Instruct"**, **"Chat"**, or **"-it"** → always use the **chat template approach**. If it is a plain base model → the **simple `pipe("text")`** approach works fine.

In [4]:
# ══════════════════════════════════════════════════════════════
# EXAMPLE 1: A simple Bot with Strict Persona
# ══════════════════════════════════════════════════════════════

import torch
from transformers import pipeline

# Step 1: Create the pipeline by specifying appropriate chat/instruct model fine-tuned to follow role-based conversations (REQUIRES the chat template format)
pipe = pipeline(
                "text-generation",
                model      = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                dtype = torch.bfloat16,   # load in bf16 to save memory
                device_map  = "auto"            # use GPU if available
                )

# Step 2: Define the conversation as role-based messages (a list of dictionaries having three roles: "system", "user", "assistant" 
messages = [
            {"role"   : "system", "content": "You are a friendly chatbot who always responds in the style of a pirate"},
            {"role"   : "user", "content": "How many helicopters can a human eat in one sitting?"},
            ]

# Step 3: apply_chat_template() converts the role-based messages list into a single formatted string with the model's special tokens
# Every instruct model has its OWN template — apply_chat_template(), automatically uses the correct one for the loaded model
prompt = pipe.tokenizer.apply_chat_template(
                                        messages,
                                        tokenize              = False,  # return string not token IDs
                                        add_generation_prompt = True    # add <|assistant|> at the end so the model knows to start generating HERE
                                        )
# Step 4: Generate the response by passing the formatted prompt string to the pipeline
outputs = pipe(
                prompt,
                max_new_tokens = 256,  # only generate this many NEW tokens (not counting prompt)
                do_sample      = True, # use sampling (creative responses)
                temperature    = 0.7,  # moderately creative (0=deterministic, 1=very random)
                top_k          = 50,   # consider only top 50 tokens at each step
                top_p          = 0.95  # nucleus sampling — consider tokens up to 95% probability mass
                )

# outputs is a list of dicts — [{"generated_text": "..."}]
print(outputs[0]["generated_text"]) # contains the FULL text (prompt + response)


Device set to use mps


<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
How many helicopters can a human eat in one sitting?</s>
<|assistant|>
I do not have information on the specific number of helicopters a human can eat in one sitting. However, the average daily caloric intake of an adult human ranges from 2,000 to 2,500 calories, which is approximately 2,500 to 3,500 calories for a person weighing 180 pounds. It is recommended to consume a balanced diet, including fruits, vegetables, lean proteins, and whole grains, to meet your daily caloric needs.


In [6]:
# ══════════════════════════════════════════════════════════════
# EXAMPLE 2: No optional System Prompt (user message only)
# ══════════════════════════════════════════════════════════════
messages = [{"role": "user", "content": "Describe Chat Templates in simple terms."}]
prompt  = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
print(outputs[0]["generated_text"])

<|user|>
Describe Chat Templates in simple terms.</s>
<|assistant|>
Chat templates are pre-built and customizable templates used in chatbots to streamline conversations between the bot and the user. They can include pre-selected responses to common questions, relevant information, and automated answers based on user input. Chat templates can be customized to match the branding and tone of the chatbot, making it more engaging and consistent. They can also help to reduce errors and improve efficiency, especially in longer conversations or interactions. Chat templates can be set up using different tools, such as BotPress, Dialogflow, or Zapier, and can be accessed and customized by chatbot developers.


In [7]:
# ══════════════════════════════════════════════════════════════
# EXAMPLE 3: Code Generation Assistant
# ══════════════════════════════════════════════════════════════
messages = [
            {"role"   : "system","content": "You are an expert Python programmer. Always provide clean, commented code with brief explanations."},
            {"role"   : "user", "content": "Write a function that checks if a string is a palindrome."},
            ]
prompt  = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=300, do_sample=True, temperature=0.3)     # low temperature → more precise/consistent code
print(outputs[0]["generated_text"])

<|system|>
You are an expert Python programmer. Always provide clean, commented code with brief explanations.</s>
<|user|>
Write a function that checks if a string is a palindrome.</s>
<|assistant|>
Here's a Python function that checks if a string is a palindrome:

```python
def is_palindrome(s):
    """
    Check if a given string is a palindrome.

    :param s: The string to check.
    :return: True if the string is a palindrome, False otherwise.
    """
    s = s.lower()
    if len(s) < 2:
        return True
    else:
        return s == s[::-1]
```

Here's an example usage:

```python
print(is_palindrome("racecar"))  # True
print(is_palindrome("racecar"))  # False
print(is_palindrome("A man, a plan, a canal: Panama!"))  # True
print(is_palindrome("A man, a plan, a canal: Panama!"))  # False
```

In this function, we first check if the length of the string is less than or equal to 2. If it is, we return True because the string is a palindrome. Otherwise, we reverse the string and c

In [8]:
# ══════════════════════════════════════════════════════════════
# EXAMPLE 4: Multi-Turn Conversation (Chat History)
# ══════════════════════════════════════════════════════════════
# Include previous turns to give the model conversation context
# assistant turns = previous model responses
messages = [
            {"role"   : "system", "content": "You are a helpful Python tutor for beginners."},
            {"role"   : "user", "content": "What is a list in Python?"},                       # turn 1 — user
            {"role"   : "assistant", "content": "A list is an ordered collection of items."},  # turn 1 — model's previous response
            {"role"   : "user", "content": "Can you show me an example?"}                      # turn 2 — user follow-up
    # No assistant entry here — model will generate turn 2 response only
]
prompt  = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
print(outputs[0]["generated_text"])

<|system|>
You are a helpful Python tutor for beginners.</s>
<|user|>
What is a list in Python?</s>
<|assistant|>
A list is an ordered collection of items.</s>
<|user|>
Can you show me an example?</s>
<|assistant|>
Sure! Here's an example:

```python
my_list = [1, 2, 3, 4, 5]
```

In this example, `my_list` is a list containing numbers 1 through 5.

```python
my_list[0]  # Returns 1
my_list[-1] # Returns 5
my_list[1:]  # Returns [2, 3, 4]
```

In the first example, we're accessing the first two elements of the list. In the second example, we're accessing the elements starting at index `1` and ending at index `3`. In the third example, we're accessing the first three elements of the list.
